<a href="https://colab.research.google.com/github/Mayureshnema04/Mayur786/blob/main/Simple_RAG_Ollama_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple RAG App — Ollama + FAISS (Notebook Version)

A 100% open-source, fully local Retrieval-Augmented Generation (RAG) app —
no API keys, no cloud account, no cost.

This notebook is the same 6-step pipeline from the workshop slides, with an
interactive UI built right into the notebook using `ipywidgets`:

1. Upload a document
2. Split it into small chunks
3. Turn each chunk into an embedding (a list of numbers) using **Ollama**
4. Store those embeddings in a **FAISS** index
5. When you ask a question, embed the question and find the closest chunks
6. Send those chunks + your question to an Ollama chat model to get an answer

Run the cells top to bottom once, then use the widgets at the bottom to
upload a document, adjust parameters, and ask questions — no need to re-run
cells after that.


## Prerequisites

Before running this notebook:

1. **Install Ollama** from [ollama.com/download](https://ollama.com/download) and make sure it's running.
2. **Pull the two models** this notebook uses (run in a terminal, not in this notebook):

```bash
ollama pull llama3.2
ollama pull nomic-embed-text
```

- `llama3.2` — the chat model that generates answers
- `nomic-embed-text` — the embedding model that turns text into vectors

You can swap in other models later (e.g. `mistral`, `phi3`, `all-minilm`) using the text boxes in the UI below — just `ollama pull` them first.


In [ ]:
# Install the Python packages this notebook needs (safe to re-run)
%pip install -q ollama faiss-cpu pypdf python-docx numpy ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 131.6 MB/s eta 0:00:00


### Run Ollama Server in Colab

To resolve the `ConnectionError`, we need to install and run Ollama directly within the Colab environment. This command downloads and installs Ollama, then starts the server as a background process.

In [ ]:
# Install zstd, then Ollama in Colab and start the server
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5 # Give Ollama server a moment to start

# Run 'ollama' from the command line to verify the installation and list running models
!ollama list

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information...

### Pull Ollama Models

Now that Ollama is running in Colab, you need to pull the required models (`llama3.2` and `nomic-embed-text`) into this Colab environment. This might take a few minutes depending on your connection.

In [ ]:
# Pull the models used by the notebook
!/usr/local/bin/ollama pull llama3.2
!/usr/local/bin/ollama pull nomic-embed-text

After these cells have finished running, you should be able to run the UI cell (`584802b9`) and proceed with uploading documents and asking questions without the `ConnectionError`.

In [ ]:
import io

import numpy as np
import faiss
import ollama
import ipywidgets as widgets
from IPython.display import display, clear_output

from pypdf import PdfReader
from docx import Document as DocxDocument

print("Imports OK")

Imports OK


## Step 1 — Read text out of the uploaded file

Handles PDF, DOCX, and plain TXT files. `file_bytes` is the raw file content
(as bytes); `filename` is used only to decide which reader to use.


In [ ]:
def extract_text(file_bytes, filename):
    """Turn a PDF / DOCX / TXT file's raw bytes into one big string of text."""
    name = filename.lower()

    if name.endswith(".pdf"):
        reader = PdfReader(io.BytesIO(file_bytes))
        pages = [page.extract_text() or "" for page in reader.pages]
        return "\n\n".join(pages)

    if name.endswith(".docx"):
        doc = DocxDocument(io.BytesIO(file_bytes))
        paragraphs = [p.text for p in doc.paragraphs]
        return "\n\n".join(paragraphs)

    # otherwise assume it's a plain .txt file
    return file_bytes.decode("utf-8", errors="ignore")

## Step 2 — Split the text into overlapping chunks

We split text into pieces of roughly `chunk_size` characters, each one
overlapping the previous by `overlap` characters, so context isn't lost right
at the cut points.


In [ ]:
def chunk_text(text, chunk_size, overlap):
    text = text.strip()
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap  # step forward, but re-include the overlap

    return chunks

## Step 3 — Turn text into embeddings using Ollama

An embedding is a list of numbers that captures a piece of text's *meaning*.
We call Ollama once per chunk (and once per question, later).


In [ ]:
def embed_text(text, embed_model):
    response = ollama.embeddings(model=embed_model, prompt=text)
    return response["embedding"]


def embed_chunks(chunks, embed_model, on_progress=None):
    vectors = []
    for i, chunk in enumerate(chunks):
        vectors.append(embed_text(chunk, embed_model))
        if on_progress is not None:
            on_progress(i + 1, len(chunks))
    return np.array(vectors, dtype="float32")

## Step 4 — Build a FAISS index so we can search the vectors fast

We normalize each vector to length 1, then use an inner-product index.
Inner product of two normalized vectors = cosine similarity, the standard
way to compare embeddings for "how similar are these?".


In [ ]:
def build_faiss_index(vectors):
    normalized = vectors.copy()
    faiss.normalize_L2(normalized)

    dimension = normalized.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(normalized)
    return index

## Step 5 — Retrieve the most relevant chunks for a question

In [ ]:
def retrieve_chunks(question, embed_model, index, chunks, top_k):
    query_vector = np.array([embed_text(question, embed_model)], dtype="float32")
    faiss.normalize_L2(query_vector)

    scores, positions = index.search(query_vector, top_k)

    results = []
    for score, pos in zip(scores[0], positions[0]):
        if pos == -1:
            continue
        results.append({"text": chunks[pos], "score": float(score)})
    return results

## Step 6 — Ask the LLM to answer using only the retrieved chunks

This is the grounding prompt: it tells the model to answer *only* from the
provided context, and to say "I don't know" rather than guess.


In [ ]:
GROUNDED_PROMPT = """Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on this document."

Context:
{context}

Question: {question}
"""


def generate_answer(question, retrieved_chunks, llm_model, temperature):
    context = "\n\n---\n\n".join(c["text"] for c in retrieved_chunks)
    prompt = GROUNDED_PROMPT.format(context=context, question=question)

    response = ollama.chat(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": temperature},
    )
    return response["message"]["content"]

## Interactive UI

Run this cell to display the UI. It works the same way as the standalone
Streamlit app from class, just embedded in the notebook:

- Upload a document and click **Process Document**
- Adjust the sliders (chunk size, overlap, top-k, temperature) any time — you
  don't need to re-run any code cells above, just re-click the buttons
- Ask a question and click **Ask** to see the answer and the exact chunks it was grounded in


In [ ]:
import io

import numpy as np
import faiss
import ollama
import ipywidgets as widgets
from IPython.display import display, clear_output

from pypdf import PdfReader
from docx import Document as DocxDocument

# --- Widgets: models + upload --- # Re-added imports to prevent potential issues
embed_model_box = widgets.Text(value="nomic-embed-text", description="Embed model:", style={"description_width": "120px"})
llm_model_box = widgets.Text(value="llama3.2", description="Chat model:", style={"description_width": "120px"})
uploader = widgets.FileUpload(accept=".pdf,.docx,.txt", multiple=False, description="Choose file")

# --- Widgets: parameters ---
chunk_size_slider = widgets.IntSlider(value=500, min=200, max=2000, step=100, description="Chunk size:", style={"description_width": "120px"})
chunk_overlap_slider = widgets.IntSlider(value=50, min=0, max=400, step=25, description="Overlap:", style={"description_width": "120px"})
top_k_slider = widgets.IntSlider(value=3, min=1, max=10, step=1, description="Top-k:", style={"description_width": "120px"})
temperature_slider = widgets.FloatSlider(value=0.2, min=0.0, max=1.0, step=0.1, description="Temperature:", style={"description_width": "120px"})

# --- Widgets: actions + output areas ---
process_button = widgets.Button(description="Process Document", button_style="primary")
process_output = widgets.Output()

question_box = widgets.Text(placeholder="Type your question here", description="Question:", style={"description_width": "120px"}, layout=widgets.Layout(width="500px"))
ask_button = widgets.Button(description="Ask", button_style="success")
answer_output = widgets.Output()

# --- State: remembers the processed document between button clicks ---
state = {"chunks": None, "index": None, "embed_model_used": None, "doc_name": None}


def on_process_clicked(b):
    with process_output:
        clear_output()
        if not uploader.value: # Check if dictionary is empty
            print("Please choose a file first.")
            return

        # Extract the single uploaded file data from the dictionary
        # uploader.value is a dict: {filename: {'content': b'...'}}
        uploaded = list(uploader.value.values())[0]
        filename = list(uploader.value.keys())[0]
        file_bytes = uploaded["content"]

        print(f"Reading {filename}...")
        text = extract_text(file_bytes, filename)
        if not text.strip():
            print("Couldn't find any text in this file. Try a different document.")
            return

        chunks = chunk_text(text, chunk_size_slider.value, chunk_overlap_slider.value)
        print(f"Split into {len(chunks)} chunks. Generating embeddings...")

        def progress(done, total):
            if done % max(1, total // 10) == 0 or done == total:
                print(f"  embedded {done}/{total} chunks")

        vectors = embed_chunks(chunks, embed_model_box.value, on_progress=progress)
        index = build_faiss_index(vectors)

        state["chunks"] = chunks
        state["index"] = index
        state["embed_model_used"] = embed_model_box.value
        state["doc_name"] = filename

        print(f"\nDone! '{filename}' is ready — {len(chunks)} chunks indexed.")


def on_ask_clicked(b):
    with answer_output:
        clear_output()
        if state["chunks"] is None:
            print("Upload and process a document first.")
            return
        if state["embed_model_used"] != embed_model_box.value:
            print(
                "Warning: you changed the embedding model after processing this "
                "document. Click 'Process Document' again first."
            )
            return
        if not question_box.value.strip():
            print("Type a question first.")
            return

        print("Retrieving relevant chunks...")
        retrieved = retrieve_chunks(
            question_box.value, embed_model_box.value, state["index"], state["chunks"], top_k_slider.value
        )

        print("Generating answer...\n")
        answer = generate_answer(question_box.value, retrieved, llm_model_box.value, temperature_slider.value)

        print("ANSWER:")
        print(answer)
        print("\n" + "-" * 60)
        print(f"Grounded in {len(retrieved)} chunks:\n")
        for i, chunk in enumerate(retrieved, start=1):
            print(f"[Chunk {i} · similarity score: {chunk['score']:.3f}]")
            print(chunk["text"])
            print()


process_button.on_click(on_process_clicked)
ask_button.on_click(on_ask_clicked)

# --- Layout ---
display(widgets.HTML("<h3>1. Settings</h3>"))
display(embed_model_box, llm_model_box)
display(chunk_size_slider, chunk_overlap_slider, top_k_slider, temperature_slider)

display(widgets.HTML("<h3>2. Upload a document</h3>"))
display(uploader, process_button, process_output)

display(widgets.HTML("<h3>3. Ask a question</h3>"))
display(question_box, ask_button, answer_output)

HTML(value='<h3>1. Settings</h3>')

Text(value='nomic-embed-text', description='Embed model:', style=DescriptionStyle(description_width='120px'))

Text(value='llama3.2', description='Chat model:', style=DescriptionStyle(description_width='120px'))

IntSlider(value=500, description='Chunk size:', max=2000, min=200, step=100, style=SliderStyle(description_wid…

IntSlider(value=50, description='Overlap:', max=400, step=25, style=SliderStyle(description_width='120px'))

...

Output()

HTML(value='<h3>3. Ask a question</h3>')

Text(value='', description='Question:', layout=Layout(width='500px'), placeholder='Type your question here', s…

Button(button_style='success', description='Ask', style=ButtonStyle())

Output()

## Parameters to experiment with

| Parameter | What it does | Try this |
|---|---|---|
| **Embedding model** | Converts text into vectors for similarity search | Try `all-minilm` (smaller/faster) after `ollama pull all-minilm` |
| **Chat model** | Generates the final answer | Try `mistral` or `phi3` after pulling them |
| **Chunk size** | How many characters per chunk | Shrink it and watch retrieval get more precise but lose surrounding context |
| **Chunk overlap** | Shared text between consecutive chunks | Set it to 0 and see if answers near chunk boundaries get worse |
| **Top-k** | How many chunks are retrieved per question | Increase it for broad questions, decrease it for narrow ones |
| **Temperature** | Randomness of the generated answer | Set it near 1.0 and ask the same question twice — compare the answers |

## Troubleshooting

| Problem | Fix |
|---|---|
| `ConnectionError` when processing or asking | Ollama isn't running — open the Ollama app, or run `ollama serve` in a terminal |
| `model not found` error | You haven't pulled that model yet — run `ollama pull <model-name>` |
| `uploader.value` is empty when clicking Process | Make sure a file is actually selected in the upload widget first |
| Processing is slow | Normal on a laptop CPU, especially for large PDFs — try a smaller document first |
